# Part 5 - Validation

Four checks confirm the construction. One benchmarks the representation, two test
properties the build guarantees, and one compares the responsive multilayer graph
against an *independent* mechanistic network solved by the study's own CARNIVAL
pipeline. None reports a biological result.

In [1]:
import numpy as np
import pandas as pd
import polars as pl

import annnet as an
import uc2b_common as uc

G = uc.load()
E = G.views.edges().to_pandas()
E = E[E.ml_kind == "intra"].copy()


def time_of(repr_str):
    inner = repr_str[repr_str.find("(", 2):]
    for t in uc.ALL_TIMES:
        if f"'{t}'" in inner:
            return t
    return None

E["t"] = E["source"].map(time_of)
E["src"] = E["source"].map(uc.bare_vid)
E["tgt"] = E["target"].map(uc.bare_vid)

## Native hyperedges vs. clique expansion

A tool without hyperedges expands each hyperedge into n(n-1)/2 pairwise edges, and
drops the signed metabolic stoichiometry, which has no pairwise encoding. The cell
reports both costs.

In [2]:
an.to_nx(G)  # materialises fine
clique = sum(n * (n - 1) // 2 for n in
             (len(set(s.get("head", [])) | set(s.get("tail", [])) | set(s.get("members", [])))
              for s in G.hyperedge_definitions.values()) if n >= 2)
native = G.global_count("edges")
hyper = len(G.hyperedge_definitions)
print(f"AnnNet native edges   : {native:,}  ({hyper:,} hyperedges)")
print(f"clique-expanded edges : {native - hyper + clique:,}  "
      f"({(native - hyper + clique) / native:.2f}x, and signed stoichiometry lost)")

AnnNet native edges   : 63,891  (16,417 hyperedges)
clique-expanded edges : 319,826  (5.01x, and signed stoichiometry lost)


## Construction invariants and the CARNIVAL comparison

The **responsive invariant** confirms the 0-hop gate: every intra-layer signaling
or regulatory edge has *both* endpoints responsive at its own timepoint. The **time
invariant** confirms every time-coupling edge links the same entity across
consecutive layers. The **CARNIVAL comparison** then asks how much of the study's
independently-solved early/late network the responsive graph already contains: the
graph is built from raw prior knowledge plus the response gate, with no CARNIVAL
optimisation, so overlap is evidence the gate selects the right subnetwork.

In [3]:
resp = pl.read_parquet(str(uc.RESPONSIVE)).to_pandas()
resp_at = uc.responsive_by_time(resp)

# Responsive invariant: both endpoints of every signaling/regulatory edge respond at t.
bad = 0
for r in E[E.edge_kind.isin(["signaling", "regulatory"])].itertuples(index=False):
    R = resp_at.get(r.t, set())
    if r.src.split(":", 1)[1] not in R or r.tgt.split(":", 1)[1] not in R:
        bad += 1
print(f"responsive invariant : {bad} violating edges (expect 0)")

# Time invariant: coupling_time links one symbol across consecutive timepoints.
ok = tot = 0
tc = G.views.edges().to_pandas()
tc = tc[tc.edge_kind == "coupling_time"]
idx = {t: i for i, t in enumerate(uc.TIMES)}
for r in tc.itertuples(index=False):
    tot += 1
    a, b = uc.bare_vid(r.source), uc.bare_vid(r.target)
    ta, tb = time_of(r.source), time_of(r.target)
    if a == b and idx.get(tb, -9) - idx.get(ta, -9) == 1:
        ok += 1
print(f"time invariant       : {ok}/{tot} coupling edges consecutive & same-entity")

# CARNIVAL early/late recovery (undirected symbol pairs).
carn = pl.read_parquet(str(uc.CARNIVAL)).to_pandas()
E["sa"] = E.src.str.split(":").str[1]
E["sb"] = E.tgt.str.split(":").str[1]


def graph_pairs(times):
    sub = E[E.t.isin(times) & E.edge_kind.isin(["signaling", "regulatory"])]
    return {frozenset((a, b)) for a, b in zip(sub.sa, sub.sb) if a and b}

print("\nCARNIVAL recovery (independent solved network):")
for net, times in [("early", uc.EARLY_TIMES), ("late", uc.LATE_TIMES)]:
    gp = graph_pairs(times)
    cp = {frozenset((r.source, r.target)) for r in carn[carn.network == net].itertuples()}
    hit = len(cp & gp)
    print(f"  {net:5s}: {hit}/{len(cp)} edges present in graph ({hit/len(cp):.0%})  "
          f"| graph {net} pairs: {len(gp):,}")

responsive invariant : 0 violating edges (expect 0)


time invariant       : 7660/7660 coupling edges consecutive & same-entity

CARNIVAL recovery (independent solved network):
  early: 9/149 edges present in graph (6%)  | graph early pairs: 4,579
  late : 45/190 edges present in graph (24%)  | graph late pairs: 15,149


## Null-DoRothEA permutation

The permutation test rewires the regulatory layer at random and re-runs the 96h Q1
co-regulation query, 50 times. A random layer should not recover the real
(TF, complex) pairs.

In [4]:
subunits = uc.complex_subunits(G, min_size=3)
reg96 = E[(E.edge_kind == "regulatory") & (E.t == "96h")]
tf_targets = (reg96.assign(tf=reg96.src.str.removeprefix("gene:"), g=reg96.tgt.str.removeprefix("gene:"))
              .groupby("tf")["g"].apply(set))
real = {(tf, cid) for cid, genes in subunits.items() for tf, tgts in tf_targets.items()
        if len(genes & tgts) / len(genes) >= 0.5 and len(genes & tgts) >= 2}

tfs = sorted(tf_targets.index)
genes_all = sorted({g for s in subunits.values() for g in s}.union(*tf_targets.values))
n_edges = len(reg96)
overlaps = []
for k in range(50):
    rng = np.random.default_rng(uc.SEED + k + 1)
    fake = (pd.DataFrame({"tf": rng.choice(tfs, n_edges), "g": rng.choice(genes_all, n_edges)})
            .groupby("tf")["g"].apply(set))
    fake_pairs = {(tf, cid) for cid, gs in subunits.items() for tf, ts in fake.items()
                  if len(gs & ts) / len(gs) >= 0.5 and len(gs & ts) >= 2}
    overlaps.append(len(real & fake_pairs))
print(f"real 96h (TF, complex) pairs: {len(real):,} | mean overlap with 50 nulls: {np.mean(overlaps):.2f}")

real 96h (TF, complex) pairs: 278 | mean overlap with 50 nulls: 0.02
